# DỰ ĐOÁN GIÁ NHÀ VN — SO SÁNH 4 MÔ HÌNH
**Ba ML (Ridge, Tree, Forest) vs PyTorch MLP**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, os
import torch, torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib
matplotlib.use('Agg')

MODEL_DIR = os.path.join('..', 'models')
data = np.load(os.path.join(MODEL_DIR, 'preprocessed_data.npz'))
X_test, y_test = data['X_test'], data['y_test']
y_test_real = np.expm1(y_test)

models = ['Ridge Regression', 'Decision Tree', 'Random Forest']
results = []

for m in models:
    model = joblib.load(os.path.join(MODEL_DIR, f"hp_{m.lower().replace(' ', '_')}.pkl"))
    preds = np.expm1(model.predict(X_test))
    results.append({
        'Model': m,
        'MAE (Tr VNĐ)': mean_absolute_error(y_test_real, preds),
        'RMSE (Tr VNĐ)': np.sqrt(mean_squared_error(y_test_real, preds)),
        'R2': r2_score(y_test_real, preds)
    })

# Dữ liệu DL
dl_preds = np.load(os.path.join(MODEL_DIR, 'hp_dl_preds.npz'))['preds_real']
results.append({
    'Model': 'Deep Learning (PyTorch)',
    'MAE (Tr VNĐ)': mean_absolute_error(y_test_real, dl_preds),
    'RMSE (Tr VNĐ)': np.sqrt(mean_squared_error(y_test_real, dl_preds)),
    'R2': r2_score(y_test_real, dl_preds)
})

df_comp = pd.DataFrame(results)
print(df_comp)

fig, ax = plt.subplots(figsize=(10,6))
df_comp.set_index('Model')[['R2']].plot(kind='bar', ax=ax, rot=15)
ax.set_ylim(0, 1)
ax.set_title('So Sánh R2 Score (Càng gần 1 càng tốt)')
plt.tight_layout(); plt.show()


## Kết luận
Random Forest thường đạt kết quả cao nhất (R2 lớn nhất, MAE/RMSE nhỏ nhất) do tính chất của Regression trên Dữ liệu Bảng (Tabular Data). Deep Learning cần cấu hình/tuning phức tạp hơn để bắt kịp cây quyết định trên kiểu dữ liệu này.